# Reporte cambios en ML y Datos

### 1.Archivo: notebooks/01_EDA.ipynb
Nombre del error: Rutas absolutas locales del sistema operativo (Absolute Local Paths).

Explicación de lo que tenemos: Al cargar el dataset crudo y exportar el dataset limpio, se utilizan rutas absolutas vinculadas a la estructura de carpetas local del ordenador de un desarrollador específico (C:/Users/Usuario/...). Esto rompe la ejecución del notebook inmediatamente al ser compartido con el resto del equipo o ejecutado en otra máquina.

Explicación del cambio a realizar: Sustituir todas las rutas absolutas por rutas relativas basadas en la raíz del proyecto para asegurar la portabilidad del código.

-Líneas de código que se modifican (Celda de Ingesta y Celda de Exportación):

    # Celda 2 (Lectura)
    df = pd.read_csv("C:/Users/Usuario/Desktop/synthetic_data_RAW.csv")

    # Celda 26 (Exportación final)
    # df_balanced.to_csv("C:/Users/Usuario/Desktop/the bridge/github/Desafio_Grupo1/data/synthetic_fin_data_CLEAN.csv", index = False)

-Líneas de código que estarían mejor:

    # Lectura portable utilizando rutas relativas
    df = pd.read_csv("../data/synthetic_data_RAW.csv")

    # Exportación portable al directorio de datos del proyecto
    df_balanced.to_csv("../data/synthetic_fin_data_CLEAN.csv", index=False)

### 2.Archivo: notebooks/01_EDA.ipynb
Nombre del error: Ausencia de variables de comportamiento temporal (Missing Velocity Features).

Explicación de lo que tenemos: El análisis exploratorio y posterior procesamiento asume que las transacciones son eventos independientes aislados. No calcula acumulados temporales por usuario (frecuencia, volumen de dinero en ventanas de tiempo de 1 hora o 24 horas), lo que dejará ciego al modelo ante ataques automatizados por scripts en la Ronda 2.

Explicación del cambio a realizar: Implementar ingeniería de características temporal (Velocity Features) agrupando por emisor (nameOrig) para detectar ráfagas sospechosas de transacciones consecutivas.


Líneas de código que estarían mejor (Añadir en el pipeline de transformación de datos):

    # Conversión y ordenación temporal para cálculo de histórico móvil
    df['timestamp'] = pd.to_datetime(df['step'], unit='h', origin=pd.Timestamp('2026-05-01'))
    df = df.sort_values(by=['nameOrig', 'timestamp'])

    # Característica de velocidad: Conteo de transacciones del mismo origen en la última hora
    df['tx_count_last_hour'] = df.groupby('nameOrig')['timestamp'].idxmax().apply(
        lambda x: df.loc[:x].groupby('nameOrig')['timestamp'].rolling('1h').count().iloc[-1]
    ) # Ajustar según rendimiento para optimizar la ventana móvil

### 3.Archivo: notebooks/02_ML.ipynb
Nombre del error: Fragmentación de artefactos de producción (Fragmented Serialized Artifacts).

Explicación de lo que tenemos: Al finalizar el entrenamiento del modelo ganador (XGBoost), el script exporta el pipeline de preprocesamiento/modelo por un lado (xgb_fraud_pipeline.joblib) y el umbral de decisión óptimo por otro (best_threshold.joblib). Esto obliga a la API a gestionar múltiples archivos asíncronos y eleva el riesgo de desajustes de versiones en producción.

Explicación del cambio a realizar: Consolidar todos los componentes del motor de inferencia (Pipeline + Umbral de decisión) en un único artefacto empaquetado como un diccionario estructurado de Python.

Líneas de código que se modifican porque están mal (Celda 12 - Guardar modelo final):

    joblib.dump(pipe_xgb, '../models/xgb_fraud_pipeline.joblib')
    joblib.dump(best_threshold, '../models/best_threshold.joblib')

Líneas de código que estarían mejor:

    # Empaquetamos el pipeline y su metadato operativo en una única estructura atómica
    fraud_engine_artifact = {
        "pipeline": pipe_xgb,
        "decision_threshold": float(best_threshold),
        "model_version": "1.0.0-round1"
    }
    joblib.dump(fraud_engine_artifact, '../models/xgb_fraud_engine.joblib')
    print("✅ Artefacto único consolidado con éxito en models/xgb_fraud_engine.joblib")